# Extract Pixel Values at Points and Zonal Statistics on Polygons from Geokuup

This notebook shows how to extract pixel values at given points or zonal statistics on given polygons from Cloud-Optimised-Geotiffs (COG) stored in the Geokuup.
- Step 1: Import libraries
- Step 2: Define helper functions
- Step 3: Set region/index/time settings
- Step 4: Load AOI geometries
- Step 5: Run point extraction
- Step 6: Run zonal extraction
- Step 7: Save outputs

## Config

First, we need to import python libraries and define functions that can be used to direct us to the COG file URL based on the given criteria. We also need to define functions to perform pixel value and zonal statistic extraction.

### Import libraries

In [22]:
import os
import requests
import geopandas as gpd
from rasterstats import point_query, zonal_stats

### Define helper utilities

- `get_season_dates`: maps season name to date range.
- `build_url`: creates the COG tile URL using parameters.
- `exists`: checks remote URL availability before reading.

In [34]:
# function to get the start and end dates for a given season and year
def get_season_dates(year, season):
    if season == "spring":
        return f"{year}-04-01", f"{year}-05-31"
    elif season == "summer":
        return f"{year}-06-01", f"{year}-08-31"
    elif season == "autumn":
        return f"{year}-09-01", f"{year}-10-31"
    else:
        raise ValueError(season)

# function to build the URL for the COG file based on the parameters
def build_url(region, sensor, index, year, start, end, composite):
    # allow both single values and list/tuple for region and sensor
    if isinstance(region, (list, tuple)):
        if not region:
            raise ValueError("region list must not be empty")
        region = region[0]
    if isinstance(sensor, (list, tuple)):
        if not sensor:
            raise ValueError("sensor list must not be empty")
        sensor = sensor[0]
    if isinstance(composite, (list, tuple)):
        if not composite:
            raise ValueError("composite list must not be empty")
        composite = composite[0]

    region_tag = region_abbrev.get(region, region)
    sensor_tag = sensor_abbrev.get(sensor, sensor)

    return (
        f"{BASE}/{region}/{sensor}/{index}/{year}/"
        f"{region_tag}_{sensor_tag}_{index}_{composite}_{start}_{end}_cog.tif"
    )

# check if the URL exists
def exists(url):
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200
    except:
        return False

### Set configuration and extraction functions

In [35]:
BASE = "https://s3.hpc.ut.ee/geokuup"

region_abbrev = {
    "estonia": "est",
    "baltics": "bal",
    "europe": "eur",
}
sensor_abbrev = {
    "sentinel1": "s1",
    "sentinel2": "s2",
    "landsat": "lsat",
}

In [36]:
# function to extract point values from the COG files
def extract_point_values(gdf, sensor, region, years, seasons, indices, composite, scale=1.0):
    gdf = gdf.to_crs(3301)
    out = gdf.copy()

    comp = composite[0] if isinstance(composite, (list, tuple)) else composite
    region_list = region if isinstance(region, (list, tuple)) else [region]
    sensor_list = sensor if isinstance(sensor, (list, tuple)) else [sensor]

    for region in region_list:
        for sensor in sensor_list:
            for year in years:
                for season in seasons:
                    start, end = get_season_dates(year, season)

                    for index in indices:
                        url = build_url(region, sensor, index, year, start, end, comp)

                        print(f"[POINT] {index} {year} {season} → {url}")

                        if not exists(url):
                            print(f"✘ Missing")
                            continue

                        try:
                            values = point_query(gdf, url)
                            if scale != 1.0:
                                values = [v * scale if v is not None else None for v in values]
                            out[f"{index}_{season}_{comp}_{year}"] = values
                        except Exception as e:
                            print(f" Failed: {index} {year} {season}")
                            print(e)

    return out

# function to extract zonal statistics from the COG files
def extract_zonal_stats(gdf, sensor, region, years, seasons, indices, composite, stats, scale=1.0):
    gdf = gdf.to_crs(3301)
    out = gdf.copy()

    comp = composite[0] if isinstance(composite, (list, tuple)) else composite
    region_list = region if isinstance(region, (list, tuple)) else [region]
    sensor_list = sensor if isinstance(sensor, (list, tuple)) else [sensor]

    for region in region_list:
        for sensor in sensor_list:
            for year in years:
                for season in seasons:
                    start, end = get_season_dates(year, season)

                    for index in indices:
                        url = build_url(region, sensor, index, year, start, end, comp)

                        print(f"[ZONAL] {index} {year} {season} → {url}")

                        if not exists(url):
                            print(f" Missing")
                            continue

                        try:
                            zs = zonal_stats(gdf, url, stats=stats)

                            for stat in stats:
                                values = [z.get(stat, None) for z in zs]
                                if scale != 1.0:
                                    values = [v * scale if v is not None else None for v in values]
                                out[f"{index}_{season}_{comp}_{year}_{stat}"] = values

                        except Exception as e:
                            print(f" Failed: {index} {year} {season}")
                            print(e)

    return out


## Load area of interest (AOI)

This part reads the point and polygon datasets from your local disk and converts them to EPSG:3301 to match the raster data CRS.

In [37]:
# comment out if you want to test with points instead of polygons

aoi_point = gpd.read_file("demo_point.gpkg")
aoi_polygon = gpd.read_file("demo_area.gpkg")

## Set parameters

In [38]:
# define parameters

sensor = [
    "sentinel2",
    #"sentinel1",
    #"landsat"
]

region = [
    "estonia",
    #"baltics",
    #"europe",
]

years = [
    #"2017",
    #"2018",
    #"2019",
    #"2020",
    #"2021",
    #"2022",
    "2023",
    #2024,
    #2025,
]

seasons = [
    "spring", 
    "summer", 
    #"autumn"
]

indices = [
     "bsi",
     "evi",
     #"fvc",
     #"lai",
     #"ndmi",
     #"ndvi",
     #"ndwi",
     #"savi,
]

# temporal composite method e.g. median summer, mean spring, etc.
composite = [
    "mean",
    "median",
    #"std",
    #"min",
    #"max",
    #"range",
]

# stats for areal/zonal summary statistics
stats = [
    #"mean",
    #"median",
    "std",
]

## Execute extraction

### Execute point extraction

- Uses `extract_point_values`
- Loops through years, seasons, and indices
- Adds a new column for each index-year-season combination

`%%time` reports runtime for profiling.

In [39]:
%%time

point_result = extract_point_values(
    aoi_point, sensor, region, years, seasons, indices, composite, scale=0.001
)

point_result.head()

[POINT] bsi 2023 spring → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/bsi/2023/est_s2_bsi_mean_2023-04-01_2023-05-31_cog.tif
[POINT] evi 2023 spring → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/evi/2023/est_s2_evi_mean_2023-04-01_2023-05-31_cog.tif
[POINT] bsi 2023 summer → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/bsi/2023/est_s2_bsi_mean_2023-06-01_2023-08-31_cog.tif
[POINT] evi 2023 summer → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/evi/2023/est_s2_evi_mean_2023-06-01_2023-08-31_cog.tif
CPU times: total: 344 ms
Wall time: 1.3 s


,id,type,geometry,bsi_spring_mean_2023,evi_spring_mean_2023,bsi_summer_mean_2023,evi_summer_mean_2023
0,1,field island,POINT (632084.69 6553633.435),0.058776,0.305763,-0.279580,0.660858
1,2,arable land,POINT (632246.002 6553289.77),0.088453,0.238079,0.017046,0.343553
2,3,water,POINT (629597.196 6549949.798),-0.139031,-0.000146,-0.224399,-0.012155
3,4,forest,POINT (631217.084 6554656.593),-0.071687,0.410868,-0.438862,0.838490
4,5,built-up,POINT (628624.995 6554793.371),0.073713,0.086997,0.084384,0.069671


### Execute zonal stats extraction

- Uses `extract_zonal_stats`
- Computes zonal stats for each polygon
- Adds columns like `ndvi_summer_median_2023_mean` where `median` indicates temporal composite method and `mean` indicates zonal statistic used

In [40]:
%%time

zonal_result = extract_zonal_stats(
    aoi_polygon, sensor, region, years, seasons, indices, composite, stats, scale=0.001
)

zonal_result.head()

[ZONAL] bsi 2023 spring → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/bsi/2023/est_s2_bsi_mean_2023-04-01_2023-05-31_cog.tif
[ZONAL] evi 2023 spring → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/evi/2023/est_s2_evi_mean_2023-04-01_2023-05-31_cog.tif
[ZONAL] bsi 2023 summer → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/bsi/2023/est_s2_bsi_mean_2023-06-01_2023-08-31_cog.tif
[ZONAL] evi 2023 summer → https://s3.hpc.ut.ee/geokuup/estonia/sentinel2/evi/2023/est_s2_evi_mean_2023-06-01_2023-08-31_cog.tif
CPU times: total: 516 ms
Wall time: 1.21 s


,id,type,geometry,bsi_spring_mean_2023_std,evi_spring_mean_2023_std,bsi_summer_mean_2023_std,evi_summer_mean_2023_std
0,1,field island,"POLYGON ((632084.69 6553652.051, 632085.015 65...",0.040951,0.047826,0.058567,0.082243
1,2,arable land,"POLYGON ((632246.002 6553308.387, 632246.327 6...",0.044293,0.050378,0.025470,0.027072
2,3,water,"POLYGON ((629597.196 6549968.415, 629597.521 6...",0.014088,0.001963,0.006326,0.001534
3,4,forest,"POLYGON ((631217.084 6554675.209, 631217.409 6...",0.031241,0.013362,0.020998,0.042610
4,5,built-up,"POLYGON ((628624.995 6554811.988, 628625.32 65...",0.012002,0.005235,0.013845,0.005100


## Save outputs

- Write GeoPackage and CSV for points or polygons
- Adjust file names as needed

#### Points

In [41]:
# export result as gpkg
point_result.to_file("points_output.gpkg", driver="GPKG")

# export result as csv
point_result.to_csv("points_output.csv", index=False)

#### Polygons

In [42]:
polygons_result = zonal_result

# export result as gpkg
polygons_result.to_file("polygons_output.gpkg", driver="GPKG")

# export result as csv
polygons_result.to_csv("polygons_output.csv", index=False)